## packages

In [6]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

# models 
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# metrics
from sklearn.metrics import (
    confusion_matrix,
    roc_auc_score
)

## data and preprocess

In [7]:
# load in data
data = pd.read_csv('simdata.csv')

# split data to train and test
train, test = train_test_split(
    data, test_size = 0.2, random_state = 42, 
)

## bootstrap method

In [11]:
# prep for bootstrapping
# make the 6 buckets -> A0, A1, B0, B1, C0, C1

# define categories and classes
planets = ["A", "B", "C"] #
classes = [0, 1]

# make empty list for buckets
buckets = {}

# make buckets
for p in planets:
    for c in classes:
        buckets[f"{p}{c}"] = train[    # only training data
            (train["planet"] == p) &
            (train["target"] == c)
        ].copy()

for name, bucket in buckets.items():
    print(name, len(bucket))


A0 2239
A1 123
B0 4549
B1 296
C0 781
C1 12


In [13]:
# bootstrapping method

# bootstrapping the buckets

boot_buckets = {}

for name, df in buckets.items():
    boot_buckets[name] = df.sample(
        n=1000,          # size of the boot buckets, placeholder for now
        replace=True,       # resample with replacement
        random_state=42     
    )

print("Original buckets:")
for name, df in buckets.items():
    print(f"{name}: {len(df)}")

print("\nBootstrapped buckets:")
for name, df in boot_buckets.items():
    print(f"{name}: {len(df)}")


# combine the boot buckets
boot_train = pd.concat(
    boot_buckets.values(),
    ignore_index=True
)
   
# shuffle for fitting model
boot_train = boot_train.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# print(boot_train["target"].value_counts())
# print(boot_train["planet"].value_counts())
#print(pd.crosstab(boot_train["planet"], boot_train["target"]))


Original buckets:
A0: 2239
A1: 123
B0: 4549
B1: 296
C0: 781
C1: 12

Bootstrapped buckets:
A0: 1000
A1: 1000
B0: 1000
B1: 1000
C0: 1000
C1: 1000


## modeling